# DIOS 接続確認

**文字入力・コードの貼り付けは不要です。下の「▶」を押してください。**

Googleの許可画面が出たら、会社のGoogleアカウントで内容を確認して承認してください。初回は「実行前の警告」「実行する」が表示される場合もあります。このノートはDIOS開発用に作成したものです。

実行する内容は、対象プロジェクトの照合、課金状態・権限・API・既存環境の読取確認です。確認結果1件だけを**自分のGoogle Driveに新規保存**します。GCPの設定変更、サーバーやデータベースの作成、課金の有効化、銀行操作、既存ファイルの上書き、共有公開は行いません。

Google Colabの認証画面では、実行環境からCloudやDriveにアクセスする権限が表示されることがあります。これはGoogle Colabへの認証であり、このチャットへのGoogle Cloud遠隔操作権限の付与ではありません。今回のプログラムは上記の確認と結果保存だけを行います。認証情報をチャットやGitHubへ送信しません。

完了後はチャットに **「確認終わった」** とだけ返信してください。本番DIOSの起動が完了するボタンではありません。

In [ ]:
#@title ▶ DIOSの接続状態を確認する（設定は変更しません）
# このセルが行うこと: Google公式認証 → GCPの読取確認 → 確認結果だけ自分のDriveへ新規保存。
# 本番起動・リソース作成・課金設定・IAM変更・既存ファイルの上書き・共有公開は行いません。
import hashlib
import json
import re
import types
import uuid
from datetime import datetime, timezone

SOURCE_COMMIT = "eedf544b3acecc282b0d212ed03cbccd5fb4e319"
SOURCE_URL = "https://raw.githubusercontent.com/sakamoto55-boop/shacho-ai-control-tower/" + SOURCE_COMMIT + "/tools/preflight/gcp_readonly.py"
SOURCE_SHA256 = "66998536433b8d340b43a5a49235f5befe20621ccf5ca7bf7c604592ebe6e53f"
DRIVE_URL = "https://www.googleapis.com/upload/drive/v3/files?uploadType=multipart&fields=id,name"
PROJECT_ID = "lcc-command"
PROJECT_NUMBER = "165404921774"


def load_checker(http):
    response = http.get(SOURCE_URL, timeout=30, allow_redirects=False)
    if response.status_code != 200 or len(response.content) > 100000:
        raise RuntimeError("CHECKER_DOWNLOAD_FAILED")
    if hashlib.sha256(response.content).hexdigest() != SOURCE_SHA256:
        raise RuntimeError("CHECKER_INTEGRITY_FAILED")
    module = types.ModuleType("dios_readonly_pinned")
    exec(compile(response.content, "dios_readonly_pinned.py", "exec"), module.__dict__)
    return module


def save_report(http, credentials, report):
    # 1回だけ新規作成。失敗・応答不明時に自動再送して重複させません。
    run_id = uuid.uuid4().hex
    name = "DIOS_PREFLIGHT_lcc-command_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + run_id[:8] + ".json"
    metadata = {"name": name, "mimeType": "application/json",
                "appProperties": {"dios_kind": "readonly_preflight", "source_commit": SOURCE_COMMIT, "run_id": run_id}}
    record = {"kind": "DIOS_READONLY_PREFLIGHT", "source_commit": SOURCE_COMMIT, "run_id": run_id, "report": report}
    boundary = "dios_" + uuid.uuid4().hex
    body = ("--" + boundary + "\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n" + json.dumps(metadata, ensure_ascii=False)
            + "\r\n--" + boundary + "\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n" + json.dumps(record, ensure_ascii=False)
            + "\r\n--" + boundary + "--\r\n").encode("utf-8")
    response = http.post(DRIVE_URL, headers={"Authorization": "Bearer " + credentials.token,
                            "Content-Type": "multipart/related; boundary=" + boundary},
                         data=body, timeout=30, allow_redirects=False)
    if response.status_code not in (200, 201):
        return None
    payload = response.json()
    if not isinstance(payload, dict) or payload.get("name") != name or not re.fullmatch(r"[A-Za-z0-9_-]{10,200}", str(payload.get("id", ""))):
        return None
    return name


def run_dios_check():
    try:
        import requests
        checker = load_checker(requests)
    except Exception:
        print("確認プログラムを安全に取得できませんでした。ここで停止しました。設定変更はありません。")
        return
    print("Googleの確認画面が出たら、会社のGoogleアカウントで承認してください。")
    print("Google Colabに認証します。このプログラムはGCPの読取と確認結果のDrive新規保存だけを実行します。")
    try:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import Request
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        credentials.refresh(Request())
        if not isinstance(credentials.token, str) or not credentials.token:
            raise RuntimeError("AUTH_REQUIRED")
    except Exception:
        print("Google認証が完了していません。許可できなかった場合は、この表示だけチャットへ送ってください。")
        return
    try:
        print("接続状態を確認中です。設定の作成・変更は行いません。")
        report = checker.collect(PROJECT_ID, PROJECT_NUMBER, checker.GoogleReader(credentials.token))
        print(checker.display(report))
        if report.get("project_verified") is not True:
            print("対象プロジェクトを確認できなかったため、Driveへ保存せず停止しました。この結果の画面を送ってください。")
            return
        try:
            saved_name = save_report(requests, credentials, report)
        except Exception:
            saved_name = None
        if saved_name:
            print("\n【確認終了】確認結果を自分のGoogle Driveに新規保存しました。共有設定は変更していません。")
            print("保存ファイル: " + saved_name)
            print("チャットに『確認終わった』とだけ返してください。結果は接続済みのDriveから読み取ります。")
        else:
            print("\nGoogle Driveへの保存は確認できませんでした。自動再送はしません。上の確認結果だけ画面で送ってください。")
        print("本番起動: 未実施。今回の完了は接続状態の確認だけです。")
    except Exception:
        print("確認を完了できませんでした。設定の作成・変更はしていません。この表示だけチャットへ送ってください。")
    finally:
        credentials = None


run_dios_check()
